In [2]:
### RAG Pipeline- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_7319/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/home/user/Desktop/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process.")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            #Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#Process all pdfs in the directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process.

Processing JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf
Loaded 7 pages

Processing btech_final_report.pdf
Loaded 51 pages

Processing sem8_end_report.pdf
Loaded 50 pages

Processing 159 Solutions - Consulting - Analyst JD v1.0.pdf
Loaded 1 pages

Total documents loaded: 109


In [15]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-05-06T20:00:13+05:30', 'title': 'PowerPoint Presentation', 'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com', 'moddate': '2026-05-06T20:00:13+05:30', 'source': '../data/pdf/JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'file_type': 'pdf'}, page_content='Campus hiring 2026\nCognizant Ace \nTeam Program'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-05-06T20:00:13+05:30', 'title': 'PowerPoint Presentation', 'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com', 'moddate': '2026-05-06T20:00:13+05:30', 'source': '../data/pdf/JD_2026 Cogni

In [19]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])
    
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [20]:
chunks=split_documents(all_pdf_documents)
chunks

Split 109 documents into 183 chunks.

Example chunk:
Content: Campus hiring 2026
Cognizant Ace 
Team Program...
Metadata: {'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-05-06T20:00:13+05:30', 'title': 'PowerPoint Presentation', 'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com', 'moddate': '2026-05-06T20:00:13+05:30', 'source': '../data/pdf/JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-05-06T20:00:13+05:30', 'title': 'PowerPoint Presentation', 'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com', 'moddate': '2026-05-06T20:00:13+05:30', 'source': '../data/pdf/JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'file_type': 'pdf'}, page_content='Campus hiring 2026\nCognizant Ace \nTeam Program'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-05-06T20:00:13+05:30', 'title': 'PowerPoint Presentation', 'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com', 'moddate': '2026-05-06T20:00:13+05:30', 'source': '../data/pdf/JD_2026 Cogni

### embedding And vectorStoreDB

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model=None
        self.__load_model()

    def __load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully Embedding dimension: {self.model.get_embedding_dimension()}.")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

embedding_manager=EmbeddingManager()
embedding_manager
        

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3591.10it/s]


Model loaded successfully Embedding dimension: 384.


### VectorStore

In [23]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client=None
        self.__initialize_store()

    def __initialize_store(self):
        try:
            print("Initializing vector store...")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF document embeddings for RAG"})
            print(f"Vector store initialized successfully. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store...")
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(ids=ids, embeddings=embeddings_list, metadatas=metadatas, documents=documents_text)
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore()
vector_store
    

Initializing vector store...
Vector store initialized successfully. Collection: pdf_documents
Existing documents in collection: 366


In [24]:
chunks

[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-05-06T20:00:13+05:30', 'title': 'PowerPoint Presentation', 'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com', 'moddate': '2026-05-06T20:00:13+05:30', 'source': '../data/pdf/JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf', 'file_type': 'pdf'}, page_content='Campus hiring 2026\nCognizant Ace \nTeam Program'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-05-06T20:00:13+05:30', 'title': 'PowerPoint Presentation', 'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com', 'moddate': '2026-05-06T20:00:13+05:30', 'source': '../data/pdf/JD_2026 Cogni

In [26]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

['Campus hiring 2026\nCognizant Ace \nTeam Program',
 '© 2026-2027 Cognizant. All rights reserved.\nAbout Cognizant\nRanked\namong the World’s Best \nCompanies by TIME\n(September 2025)\nNamed\nas one of America’s Greatest \nWorkplaces and Greatest \nWorkplaces in Tech by Fortune\n(June 2025)\nRanked\namong Newsweek’s Most \nReliable Companies\n(October 2025)\nNamed\nin Forbes’ World’s Best \nEmployers list 2025\n(October 2025)\nFeatured\nin Newsweek’s America’s \nGreatest Workplaces for \nGen Z \n(May 2025)\nRanked\namong Fortune’s America’s \nMost Innovative Companies\n(March 2026)\nAchieved\na Guinness World Record for the \nlargest online generative AI \nhackathon\n(August 2025)\nRecognized\nin the Fortune 500 list for the \n15th consecutive year and \nranked 3rd among pure IT \nconsultancies\n(June 2025)\nCertified\nas Great Place to Work® in 31 \ncountries, including India\n(March 2026)\nRecognized\nas one of the World’s Most Ethical \nCompanies® by Ethisphere\n(March 2026)',
 'J

In [27]:
embeddings=embedding_manager.generate_embeddings(texts)

### store in vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 183 texts...


Batches: 100%|██████████| 6/6 [00:07<00:00,  1.18s/it]


Generated embeddings with shape: (183, 384)
Adding 183 documents to vector store...
Documents added successfully. Total documents in collection: 549


Retriever Pipeline from VectorStore

In [28]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriver

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents based on the query

        Args:
            query: User query string
            top_k: Number of top relevant documents to retrieve
            score_threshold: Minimum similarity score for retrieved documents

        Returns:
            List of retrieved documents with metadata
        """
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
    
                retrieved_docs = []

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id, 
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold.")
            else:
                print("No documents retrieved from vector store.")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        

rag_retriever=RAGRetriever(vector_store, embedding_manager)
rag_retriever


In [29]:
rag_retriever.retrieve("What is cognizant?")

Retrieving documents for query: 'What is cognizant?' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.88it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold.


[{'id': 'doc_8679ab51_3',
  'content': 'stakeholders.\nThrough structured rotations across domains, technology stacks, and problem types, \nthe program enables you to rapidly build both depth and breadth of experience—\ndeveloping, within a short span, a portfolio of solved problems that traditionally \ntakes 5+ years to accumulate.\nJob  location:  PAN India\n© 2026-2027 Cognizant. All rights reserved.\n(Candidate must be flexible to relocate to any of \nthe Cognizant office location as per the business \nrequirement & demand).\nIf you’re looking for predictability, this isn’t the path. If you’re driven to grow \ninto an exceptional engineer and earn your place among the ace, read on.\nCognizant Ace Team',
  'metadata': {'author': 'PavithraLakshmi.Sathyanarayanan@cognizant.com',
   'creationdate': '2026-05-06T20:00:13+05:30',
   'page': 2,
   'doc_index': 3,
   'source': '../data/pdf/JD_2026 Cognizant ACE Team program_Associate and Senior Associate AI Engineer.pdf',
   'page_label': '

Integration Vectordb Context pipeline with LLM output

In [30]:
### Simple RAG pipeline using Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in the environment)

groq_api_key = os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieves relevant documents and generates answer using LLM

def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([f"Document {i+1}:\n{res['content']}" for i, res in enumerate(results)]) if results else ""
    if not context:
        return "No relevant documents found to answer the query."
    
    ## generate the answer using the GROQ LLM
    prompt=f"Use the following retrieved documents to answer the question:\n\n{context}\n\nQuestion: {query}\nAnswer:"

    response=llm.invoke(prompt)
    return response.content



In [36]:
answer=rag_simple("What is cognizant?", rag_retriever, llm)
print("Answer:", answer)

Retrieving documents for query: 'What is cognizant?' with top_k=3 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.12it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents after applying score threshold.


Answer: Cognizant is a company that offers a program called the Cognizant Ace Team, which provides a structured rotation across domains, technology stacks, and problem types to help engineers rapidly build experience and develop a portfolio of solved problems.


Enhanced RAG Pipeline Features

In [41]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    Advanced RAG function that retrieves relevant documents, filters by similarity score, and generates answer using LLM.

    Args:
        query: User query string
        retriever: RAGRetriever instance for retrieving documents
        llm: Language model instance for generating answer
        top_k: Number of top relevant documents to retrieve
        min_score: Minimum similarity score for considering retrieved documents
        return_context: Whether to return the retrieved context along with the answer"""
    
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {'answer': "No relevant context found.", 'sources': [], 'confidence':0.0, 'context':''}
    
    context = "\n\n".join([doc['content'] for doc in results])
    
    sources = [{'source':doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'score': doc['similarity_score'],
                'page':doc['metadata'].get('page', 'unknown'),
                'preview':doc['content'][:300] + '...'} for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer

    prompt = f"Use the following retrieved documents to answer the question:\n\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt)

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
    }
    if return_context:
        output['context'] = context
    
    return output

#Example usage
result = rag_advanced("Efficient Estimation of Word Representations in Vector Space", rag_retriever, llm, top_k=5, min_score=0.0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence Score:", result['confidence'])
print("Retrieved Context:", result['context'][:300])

    

Retrieving documents for query: 'Efficient Estimation of Word Representations in Vector Space' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.14it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents after applying score threshold.


Answer: The answer to the question "Efficient Estimation of Word Representations in Vector Space" is a research paper by T. Mikolov, K. Chen, G. Corrado, and J. Dean, published in 2013 as an arXiv preprint (arXiv:1301.3781). 

This paper introduces a method for efficiently estimating word representations in vector space, which is a fundamental concept in natural language processing. The authors propose a technique that allows for the creation of high-dimensional vector representations of words, capturing their semantic meanings and relationships.

The key contributions of this paper include:

1. **Word2Vec**: The authors introduce the Word2Vec algorithm, which is a framework for learning vector representations of words.
2. **Efficient estimation**: The paper presents an efficient method for estimating word representations in vector space, which is scalable to large datasets.
3. **Vector space modeling**: The authors demonstrate the effectiveness of their approach in capturing semantic 